# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a practical guide for loading and exploring the FAIR² dataset using the `mlcroissant` library. We'll cover essential data loading, exploration, and basic analysis steps following best practices for referencing Croissant entities by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema JSON-LD URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`


In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the Croissant dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Accessing dataset metadata as an object (not dict-like)
meta = dataset.metadata
print(f"Dataset name: {meta.name}\nDescription: {meta.description}\nPublished: {meta.datePublished}")

## 2. Data Overview
Review registered record sets and their fields (always using the `@id`).

The Croissant schema organizes data into *record sets* (tables or logical groupings), with each field and column also identified by a globally unique `@id`. Let's enumerate record sets and their fields/columns to plan our extraction.

In [ ]:
# List available record sets and their IDs
if hasattr(meta, 'recordSets'):
    record_sets = meta.recordSets
else:
    # Try with record_sets as plural/singular
    record_sets = getattr(meta, 'recordSet', [])

print("Available Record Sets (referenced by @id):\n")
all_recordset_ids = []
for rs in record_sets:
    print(f"- {rs['@id']} : {rs.get('name', '(no name)')}")
    all_recordset_ids.append(rs['@id'])
    # List fields/columns for each record set
    fields = rs.get('field', []) or []
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields/Columns:")
    for fld in fields:
        fid = fld['@id'] if isinstance(fld, dict) and '@id' in fld else fld
        print(f"    - {fid}")
    print()
if not all_recordset_ids:
    print('No record sets found in metadata!')


In [ ]:
# Inspect the first record set's actual sample records using its @id
if all_recordset_ids:
    chosen_record_set = all_recordset_ids[0]
    print(f'Record examples from record_set: {chosen_record_set}\n')
    for idx, record in enumerate(dataset.records(record_set=chosen_record_set)):
        print(json.dumps(record, indent=2))
        if idx >= 2:  # Show only a few samples
            break
else:
    print('No record sets found to display records.')

## 3. Data Extraction
Load data from available record sets into Pandas DataFrames for analysis (referencing everything by `@id`).

**Tip:** Use the cell above to decide which record set(s) and fields you'd like to analyze.

In [ ]:
# Extract all record sets into Pandas dataframes and inspect their columns
dataframes = {}
for record_set_id in all_recordset_ids:
    records = list(dataset.records(record_set=record_set_id))
    if not records:
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record set: {record_set_id} - columns: {df.columns.tolist()}")
    print(df.head(2))

# For the rest of the notebook, pick one record set as an example
example_record_set_id = all_recordset_ids[0] if all_recordset_ids else None

if example_record_set_id:
    print(f"\nChosen record_set for analysis: {example_record_set_id}")
    print(f"Columns: {dataframes[example_record_set_id].columns.tolist()}")
    display(dataframes[example_record_set_id].head())
else:
    print("No available records sets with data.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps—such as filtering records, normalizing numeric fields, and grouping data. All columns are referenced using their full `@id` as defined by the dataset for reproducibility.

**Example below uses one numeric field (by @id, adjust as needed) and performs normalization and grouping.**

In [ ]:
if example_record_set_id and example_record_set_id in dataframes:
    df = dataframes[example_record_set_id].copy()
    print("Available columns (by @id):\n", df.columns.tolist())
    # Try to auto-detect a candidate numeric field by checking dtype
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    # Pick the first numeric field for demonstration, or ask user to specify
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Selected numeric field for EDA: {numeric_field_id}")
        # Filter records (choose an arbitrary threshold for illustration)
        threshold = df[numeric_field_id].mean()  # e.g., mean as threshold
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric field (Z-score/standard score)
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()

        print(f"\nNormalized {numeric_field_id} (z-score):")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a categorical field (e.g., the first string column)
        group_field_candidates = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
        if group_field_candidates:
            group_field_id = group_field_candidates[0]
            print(f"\nGrouping filtered results by: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric fields found for EDA.")
else:
    print("No data available to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships directly from the DataFrame. Here, we plot the distribution of the selected numeric field, showing filtered/normalized results, and a basic group aggregation if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_record_set_id and example_record_set_id in dataframes:
    df = dataframes[example_record_set_id]
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_fields:
        field = numeric_fields[0]
        plt.figure(figsize=(7, 4))
        sns.histplot(df[field].dropna(), kde=True)
        plt.title(f'Distribution of numeric field {field}')
        plt.xlabel(field)
        plt.ylabel('Count')
        plt.show()
    # If a grouping was performed in EDA, show a barplot
    group_field_candidates = [col for col in df.columns if df[col].dtype == object and col != field]
    if group_field_candidates:
        group_field = group_field_candidates[0]
        agg_df = df.groupby(group_field)[field].mean().reset_index()
        plt.figure(figsize=(8, 4))
        sns.barplot(x=group_field, y=field, data=agg_df)
        plt.title(f'Average {field} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print('No data for visualization.')

## 6. Conclusion
We've demonstrated loading and programmatically exploring a FAIR² mass spectrometry dataset using the Croissant schema and `mlcroissant`. All operations reference fields, record sets, and columns by their `@id` per best FAIR practices. You can repeat, adapt or extend these steps for deeper analysis, downstream machine learning, or further domain exploration.